In [1]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

MCP_TIME_URL = 'http://127.0.0.1:8325/mcp'

class StreamableHttpMcpToolSource:
    def __init__(self, url: str) -> None:
        self.url = url

    async def list_tools(self):
        async with streamablehttp_client(self.url) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                return await session.list_tools()

    async def call_tool(self, name: str, arguments: dict | None = None):
        async with streamablehttp_client(self.url) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                return await session.call_tool(name, arguments or {})

mcp_time_client = StreamableHttpMcpToolSource(MCP_TIME_URL)

mcp_tools = await mcp_time_client.list_tools()
tool_names = [tool.name for tool in mcp_tools.tools]
print('MCP tools:', tool_names)
if 'get_current_date_time' not in tool_names:
    raise RuntimeError('MCP server did not expose get_current_date_time')

direct_time = await mcp_time_client.call_tool('get_current_date_time', {})
print('direct get_current_date_time:', getattr(direct_time, 'structuredContent', None) or direct_time)


MCP tools: ['get_current_date_time', 'fetch_and_convert', 'search_web']
direct get_current_date_time: {'result': '2026-07-02 01:34:23'}


In [2]:
from openai_helpers_piplines_package import (
    JsonFixPipeline,
    LoggerPipeline,
    LoopGuardPipeline,
    PipelineRequestError,
    ToolPipeline,
    chat_session,
    classify_pipeline_error,
    dict_to_pydantic_schema,
    pipelined_chat,
    attempt,
)
import os
LOG_PATH = 'demo_pipeline.log'
BASE_URL = 'http://127.0.0.1:8080/v1'
API_KEY = "none"
MODEL_NAME = ""


In [3]:
from openai import OpenAI
import json
from dataclasses import asdict

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

chat = pipelined_chat(
    client.chat.completions,
    layers=[LoggerPipeline(path=LOG_PATH), ToolPipeline(max_retries=3),LoopGuardPipeline(max_retries=3)],
)


In [ ]:

session = chat_session(
    chat,
    model=MODEL_NAME,
    role_content={'system': 'Use the MCP-tools when it possible.',},
    temperature=0.1,
    max_tokens=500,
    tool_sources=[mcp_time_client],
    return_trace=True,
)

mcp_result = await attempt(session.step(role_content={'user':'tell current date and time and current news from CNN'}))


if isinstance(mcp_result, PipelineRequestError):
    print('MCP tool demo failed:')
    print(mcp_result)
else:
    print('assistant:', mcp_result.response['choices'][0]['message'].get('content', ''))
    print('tool executions:')
    print(json.dumps([asdict(execution) for execution in mcp_result.tool_executions], indent=2, ensure_ascii=False))
    print('trace events:', [f'{event.level}/{event.action}' for event in (mcp_result.trace or [])])

In [ ]:
mcp_result.response['choices'][0]['message'].get('reasoning_content', '')
stop()

"The `fetch_and_convert` tool successfully retrieved the CNN homepage content. I can see several headlines and news items. I will extract the top stories from this content to answer the user's request.\n\nTop stories identified from the content:\n1.  **US-Iran talks**: Qatari leaders meet senior US and Iranian officials in effort to advance talks.\n2.  **China’s ethnic minorities**: China tells its minorities to integrate or face consequences with sweeping new law.\n3.  **Ocean temperatures**: Global oceans break June temperature record.\n4.  **Trump’s cryptocurrency ventures**: Trump made over a billion dollars from cryptocurrency ventures in first year back in office.\n5.  **Desert express**: UAE's first passenger rail.\n6.  **Taylor Swift**: Everything we’re seeing in New York as Swift and Kelce’s expected wedding celebrations approach.\n7.  **Empire State Building**: Couple appeared to get engaged before being taken into custody atop Empire State Building.\n8.  **USMCA**: Trump wan

In [ ]:
mcp_result = await attempt(session.step(role_content={'user':'tell detailes about Germany Shooting. How police acts?'}))

if isinstance(mcp_result, PipelineRequestError):
    print('MCP tool demo failed:')
    print(mcp_result)
else:
    print('assistant:', mcp_result.response['choices'][0]['message'].get('content', ''))
    print('tool executions:')
    print(json.dumps([asdict(execution) for execution in mcp_result.tool_executions], indent=2, ensure_ascii=False))
    print('trace events:', [f'{event.level}/{event.action}' for event in (mcp_result.trace or [])])

MCP tool demo failed:
chat.completions.create failed: Loading model

Request stage: tool_generation

Request params: model='', temperature=0.1, max_tokens=500, stream=True, stream_options={'include_usage': True}, tools=['get_current_date_time', 'fetch_and_convert', 'search_web']
Messages (8):
  [0] system: 'Use the MCP-tools when it possible.'
  [1] user: 'tell current date and time and current news from CNN'
  [2] assistant:
        The current date and time is **2026-07-02 01:25:24**.
        
        Here are the top news stories from CNN:
        
        **World & Politics**
        *   **US-Iran Talks:** Qatari leaders are meeting with senior US and Iranian officials in an effort to advance talks.
        *   **China's New Law:** China has introduced a sweeping new law telling its ethnic minorities to integrate or face consequences.
        *   **Greece Violence:** One person is dead and several others are hurt after firebombs were hurled at the homes of members of Greece's gover

In [ ]:
mcp_result.usage

AttributeError: 'PipelineRequestError' object has no attribute 'usage'